In [2]:
# --- 1. SETUP & DATA LOADING ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load your dataset
df = pd.read_csv('C:\\Users\\ibrah\\Downloads\\energydata_complete.csv')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded: 19735 rows, 29 columns


In [3]:
# --- 2. EXPLORATORY DATA ANALYSIS & PREPROCESSING ---
print(f"Dataset shape: {df.shape}")
print(df.info())
print(df.describe())

# Convert date and extract time features
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Feature correlation analysis
correlation_matrix = df.drop(columns=['date', 'rv1', 'rv2']).corr()
target_correlation = correlation_matrix['Appliances'].sort_values(ascending=False)

print("\nTop 10 features correlated with Appliances:")
print(target_correlation.head(10))

# Define features (X) and target (y)
X = df.drop(columns=['Appliances', 'date', 'rv1', 'rv2'])
y = df['Appliances']

# Train-test split (use shuffle=False if respecting time series order)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining samples: {X_train_scaled.shape[0]}, Features: {X_train_scaled.shape[1]}")
print(f"Target range: {y.min():.2f} to {y.max():.2f} Wh")
print(f"Target mean: {y.mean():.2f} Wh, std: {y.std():.2f} Wh")

Dataset shape: (19735, 29)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  object 
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 

In [4]:
# --- 3. BASELINE NEURAL NETWORK MODEL ---
def create_baseline_model(input_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)  # Output layer for regression
    ])
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    return model

baseline_model = create_baseline_model(X_train_scaled.shape[1])
baseline_model.summary()

# Train the baseline model
print("\nTraining baseline model...")
history_baseline = baseline_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_test_scaled).flatten()
test_rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
test_r2_baseline = r2_score(y_test, y_pred_baseline)
test_mae_baseline = np.mean(np.abs(y_test - y_pred_baseline))
test_mape_baseline = np.mean(np.abs((y_test - y_pred_baseline) / y_test)) * 100

print(f"\n--- Baseline Model Results ---")
print(f"Test RMSE: {test_rmse_baseline:.2f} Wh")
print(f"Test MAE: {test_mae_baseline:.2f} Wh")
print(f"Test MAPE: {test_mape_baseline:.2f}%")
print(f"Test R²: {test_r2_baseline:.3f}")
print(f"MAE/Mean Ratio: {test_mae_baseline/y_test.mean():.2%}")

C:\Users\ibrah\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,969 (15.50 KB)

 Trainable params: 3,969 (15.50 KB)

 Non-trainable params: 0 (0.00 B)


Training baseline model...
Epoch 1/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 13024.3906 - mae: 65.5936 - val_loss: 9529.3926 - val_mae: 56.6449
Epoch 2/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9846.4248 - mae: 55.4986 - val_loss: 8683.0557 - val_mae: 53.4105
Epoch 3/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9347.7764 - mae: 54.0421 - val_loss: 8330.9023 - val_mae: 52.2276
Epoch 4/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 9097.2402 - mae: 53.3030 - val_loss: 8151.1802 - val_mae: 51.6473
Epoch 5/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 8945.9443 - mae: 52.7997 - val_loss: 8043.2671 - val_mae: 51.2711
Epoch 6/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 8841.0537 - mae: 52.3756 - val_loss: 7966.2837 - val_mae: 50.9995
Epoch 7/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 8758.6162 - mae: 52.0192 - val_loss: 7906.2480 - val_mae: 50.7027
Epoch 8/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 8690.3115 - mae: 51.7095 - va

In [5]:
# --- 4. HYPERPARAMETER TUNING & MODEL IMPROVEMENT ---
def build_model(hidden_layers, neurons_per_layer, activation='relu', dropout_rate=0.2):
    model = models.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    
    for i in range(hidden_layers):
        model.add(layers.Dense(neurons_per_layer, activation=activation))
        if i < hidden_layers - 1:  # No dropout on last hidden layer
            model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Enhanced configurations to test
configs = [
    {'hidden_layers': 2, 'neurons': 64, 'dropout': 0.1},
    {'hidden_layers': 3, 'neurons': 128, 'dropout': 0.2},
    {'hidden_layers': 4, 'neurons': 64, 'dropout': 0.2},
    {'hidden_layers': 3, 'neurons': 256, 'dropout': 0.3},
    {'hidden_layers': 2, 'neurons': 128, 'dropout': 0.1}
]

tuning_results = []
histories = []

for i, config in enumerate(configs):
    print(f"\nTesting config {i+1}/{len(configs)}: {config}")
    model = build_model(config['hidden_layers'], config['neurons'], dropout_rate=config['dropout'])
    
    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.15,
        epochs=30,
        batch_size=32,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    )
    
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    tuning_results.append({**config, 'test_rmse': rmse, 'test_r2': r2})
    histories.append(history)
    
    print(f"  RMSE: {rmse:.2f} Wh, R²: {r2:.3f}")

# Display tuning results
results_df = pd.DataFrame(tuning_results)
print("\n--- Hyperparameter Tuning Results ---")
print(results_df.sort_values('test_rmse'))

# Find best configuration
best_config = results_df.loc[results_df['test_rmse'].idxmin()]
print(f"\nBest configuration: {best_config.to_dict()}")


Testing config 1/5: {'hidden_layers': 2, 'neurons': 64, 'dropout': 0.1}
  RMSE: 86.33 Wh, R²: 0.255

Testing config 2/5: {'hidden_layers': 3, 'neurons': 128, 'dropout': 0.2}
  RMSE: 82.99 Wh, R²: 0.312

Testing config 3/5: {'hidden_layers': 4, 'neurons': 64, 'dropout': 0.2}
  RMSE: 83.99 Wh, R²: 0.295

Testing config 4/5: {'hidden_layers': 3, 'neurons': 256, 'dropout': 0.3}


KeyboardInterrupt: 

In [ ]:
# --- 5. FINAL MODEL & EVALUATION ---
# Based on tuning results, define improved final architecture
final_model = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

# Enhanced optimizer with weight decay
optimizer = keras.optimizers.Adam(
    learning_rate=0.001,
    beta_1=0.9,
    beta_2=0.999,
    weight_decay=0.0001
)

final_model.compile(
    optimizer=optimizer,
    loss='huber',  # More robust to outliers than MSE
    metrics=['mae', 'mse']
)

final_model.summary()

print("\nTraining final model...")
history_final = final_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    verbose=1,
    callbacks=[
        callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=15,  # Increased patience
            restore_best_weights=True,
            min_delta=0.001
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.5, 
            patience=7,  # Changed from 5
            min_lr=0.00001
        )
    ]
)

# Final evaluation
y_train_pred_final = final_model.predict(X_train_scaled).flatten()
y_test_pred_final = final_model.predict(X_test_scaled).flatten()

# Calculate comprehensive metrics
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

train_rmse_final = np.sqrt(mean_squared_error(y_train, y_train_pred_final))
test_rmse_final = np.sqrt(mean_squared_error(y_test, y_test_pred_final))
train_mae_final = mean_absolute_error(y_train, y_train_pred_final)
test_mae_final = mean_absolute_error(y_test, y_test_pred_final)
train_mape_final = mean_absolute_percentage_error(y_train, y_train_pred_final) * 100
test_mape_final = mean_absolute_percentage_error(y_test, y_test_pred_final) * 100
train_r2_final = r2_score(y_train, y_train_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)
print(f"Training RMSE: {train_rmse_final:.2f} Wh | R²: {train_r2_final:.3f}")
print(f"Test RMSE:     {test_rmse_final:.2f} Wh | R²: {test_r2_final:.3f}")
print(f"Test MAE:      {test_mae_final:.2f} Wh | MAPE: {test_mape_final:.2f}%")
print(f"MAE/Mean Ratio: {test_mae_final/y_test.mean():.2%}")
print(f"Improvement over baseline: {(test_rmse_baseline - test_rmse_final)/test_rmse_baseline:.2%}")

# Save all model artifacts
import json
import pickle

# Save model
final_model.save('final_neural_network_model.h5')

# Save training history
with open('training_history.json', 'w') as f:
    json.dump(history_final.history, f)

# Save scaler
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
with open('feature_names.txt', 'w') as f:
    f.write('\n'.join(X.columns.tolist()))

print("\nModel artifacts saved:")
print("  - final_neural_network_model.h5")
print("  - training_history.json")
print("  - feature_scaler.pkl")
print("  - feature_names.txt")

In [ ]:
# --- 6. CROSS-VALIDATION FOR ROBUSTNESS ---
from sklearn.model_selection import KFold

print("\nPerforming 5-fold cross-validation...")

# Function to create a fresh model for each fold
def create_cv_model(input_dim):
    model = models.Sequential([
        layers.Dense(256, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    return model

# 5-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
fold_histories = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train_scaled)):
    print(f"\n--- Training Fold {fold+1}/5 ---")
    
    # Split data for this fold
    X_train_fold = X_train_scaled[train_idx]
    y_train_fold = y_train.iloc[train_idx]
    X_val_fold = X_train_scaled[val_idx]
    y_val_fold = y_train.iloc[val_idx]
    
    # Create and train model
    model = create_cv_model(X_train_scaled.shape[1])
    history = model.fit(
        X_train_fold, y_train_fold,
        validation_data=(X_val_fold, y_val_fold),
        epochs=50,
        batch_size=32,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
    )
    
    # Evaluate on validation fold
    val_pred = model.predict(X_val_fold).flatten()
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred))
    fold_r2 = r2_score(y_val_fold, val_pred)
    
    fold_results.append({
        'fold': fold+1,
        'rmse': fold_rmse,
        'r2': fold_r2,
        'epochs': len(history.history['loss'])
    })
    
    fold_histories.append(history)
    
    print(f"  Fold {fold+1}: RMSE = {fold_rmse:.2f} Wh, R² = {fold_r2:.3f}")

# Calculate cross-validation statistics
cv_results_df = pd.DataFrame(fold_results)
print("\n--- Cross-Validation Results ---")
print(cv_results_df)

mean_rmse = cv_results_df['rmse'].mean()
std_rmse = cv_results_df['rmse'].std()
mean_r2 = cv_results_df['r2'].mean()

print(f"\nMean CV RMSE: {mean_rmse:.2f} ± {std_rmse:.2f} Wh")
print(f"Mean CV R²:   {mean_r2:.3f}")
print(f"CV Stability: {std_rmse/mean_rmse:.2%} (lower is better)")


Performing 5-fold cross-validation...

--- Training Fold 1/5 ---


C:\Users\ibrah\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step   
  Fold 1: RMSE = 80.36 Wh, R² = 0.358

--- Training Fold 2/5 ---


C:\Users\ibrah\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# --- 7. FEATURE IMPORTANCE ANALYSIS ---
print("\nAnalyzing feature importance...")

# Method 1: Permutation importance
def permutation_importance(model, X, y, metric, n_repeats=5):
    """Calculate permutation importance for neural network"""
    baseline_score = metric(y, model.predict(X).flatten())
    importance_scores = []
    
    for col in range(X.shape[1]):
        X_permuted = X.copy()
        scores = []
        
        for _ in range(n_repeats):
            np.random.shuffle(X_permuted[:, col])  # Permute this column
            score = metric(y, model.predict(X_permuted).flatten())
            scores.append(score)
        
        importance = np.mean(scores) - baseline_score
        importance_scores.append(importance)
    
    return np.array(importance_scores)

# Calculate permutation importance
print("Calculating permutation importance...")
perm_importance = permutation_importance(
    final_model, 
    X_test_scaled[:1000],  # Use subset for speed
    y_test[:1000],
    metric=lambda y_true, y_pred: -mean_squared_error(y_true, y_pred),  # Negative MSE
    n_repeats=3
)

# Create importance DataFrame
importance_df = pd.DataFrame({
    'feature': X.columns,
    'permutation_importance': perm_importance
}).sort_values('permutation_importance', ascending=False)

# Method 2: Correlation with predictions
y_pred = final_model.predict(X_test_scaled).flatten()
pred_correlations = []
for i, col in enumerate(X.columns):
    corr = np.corrcoef(X_test_scaled[:, i], y_pred)[0, 1]
    pred_correlations.append(abs(corr))  # Use absolute correlation

importance_df['prediction_correlation'] = pred_correlations
importance_df['combined_importance'] = (
    importance_df['permutation_importance'].rank() + 
    importance_df['prediction_correlation'].rank()
) / 2

print("\nTop 15 Most Important Features:")
print(importance_df.head(15))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_features = importance_df.head(15).sort_values('permutation_importance', ascending=True)
plt.barh(range(len(top_features)), top_features['permutation_importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Permutation Importance (increase in MSE when shuffled)')
plt.title('Top 15 Features by Permutation Importance')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8. COMPREHENSIVE VISUALIZATION ---
print("\nGenerating comprehensive visualizations...")

# Create dashboard with multiple plots
fig = plt.figure(figsize=(20, 16))

# 1. Training History
ax1 = plt.subplot(3, 3, 1)
ax1.plot(history_final.history['loss'], label='Training Loss', linewidth=2)
ax1.plot(history_final.history['val_loss'], label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (Huber)')
ax1.set_title('Training History - Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. MAE History
ax2 = plt.subplot(3, 3, 2)
ax2.plot(history_final.history['mae'], label='Training MAE', linewidth=2)
ax2.plot(history_final.history['val_mae'], label='Validation MAE', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE (Wh)')
ax2.set_title('Training History - MAE')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Predictions vs Actual
ax3 = plt.subplot(3, 3, 3)
scatter = ax3.scatter(y_test, y_test_pred_final, alpha=0.6, 
                     c=np.abs(y_test - y_test_pred_final), cmap='viridis')
ax3.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
        'r--', linewidth=2, label='Perfect Prediction')
ax3.set_xlabel('Actual Energy (Wh)')
ax3.set_ylabel('Predicted Energy (Wh)')
ax3.set_title('Predictions vs Actual')
plt.colorbar(scatter, ax=ax3, label='Absolute Error (Wh)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Residual Plot
ax4 = plt.subplot(3, 3, 4)
residuals = y_test - y_test_pred_final
ax4.scatter(y_test_pred_final, residuals, alpha=0.6)
ax4.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax4.set_xlabel('Predicted Energy (Wh)')
ax4.set_ylabel('Residuals (Actual - Predicted)')
ax4.set_title('Residual Plot')
ax4.grid(True, alpha=0.3)

# 5. Error Distribution
ax5 = plt.subplot(3, 3, 5)
abs_errors = np.abs(residuals)
ax5.hist(abs_errors, bins=50, edgecolor='black', alpha=0.7)
ax5.axvline(x=abs_errors.mean(), color='r', linestyle='--', linewidth=2, 
           label=f'Mean: {abs_errors.mean():.1f} Wh')
ax5.set_xlabel('Absolute Error (Wh)')
ax5.set_ylabel('Frequency')
ax5.set_title('Error Distribution')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Time-based Error Analysis
ax6 = plt.subplot(3, 3, 6)
test_dates = df.loc[y_test.index, 'date']
hourly_errors = pd.DataFrame({
    'hour': test_dates.dt.hour,
    'error': abs_errors
}).groupby('hour')['error'].mean()

ax6.plot(hourly_errors.index, hourly_errors.values, marker='o', linewidth=2)
ax6.set_xlabel('Hour of Day')
ax6.set_ylabel('Mean Absolute Error (Wh)')
ax6.set_title('Error by Hour of Day')
ax6.set_xticks(range(0, 24, 3))
ax6.grid(True, alpha=0.3)

# 7. Cumulative Error Distribution
ax7 = plt.subplot(3, 3, 7)
sorted_errors = np.sort(abs_errors)
cumulative = np.arange(1, len(sorted_errors)+1) / len(sorted_errors)
ax7.plot(sorted_errors, cumulative, linewidth=3)
ax7.axhline(y=0.9, color='r', linestyle='--', alpha=0.7, 
           label=f'90%: {np.percentile(sorted_errors, 90):.1f} Wh')
ax7.axhline(y=0.5, color='g', linestyle='--', alpha=0.7, 
           label=f'50%: {np.percentile(sorted_errors, 50):.1f} Wh')
ax7.set_xlabel('Absolute Error (Wh)')
ax7.set_ylabel('Cumulative Proportion')
ax7.set_title('Cumulative Error Distribution')
ax7.legend()
ax7.grid(True, alpha=0.3)

# 8. Learning Rate History (if available)
ax8 = plt.subplot(3, 3, 8)
if 'lr' in history_final.history:
    ax8.semilogy(history_final.history['lr'], linewidth=2)
    ax8.set_xlabel('Epoch')
    ax8.set_ylabel('Learning Rate')
    ax8.set_title('Learning Rate Schedule')
else:
    # Plot validation loss vs training loss ratio instead
    loss_ratio = np.array(history_final.history['val_loss']) / np.array(history_final.history['loss'])
    ax8.plot(loss_ratio, linewidth=2)
    ax8.axhline(y=1.0, color='r', linestyle='--', alpha=0.7)
    ax8.set_xlabel('Epoch')
    ax8.set_ylabel('Val Loss / Train Loss')
    ax8.set_title('Overfitting Indicator')
ax8.grid(True, alpha=0.3)

# 9. Model Comparison (Baseline vs Final)
ax9 = plt.subplot(3, 3, 9)
models_comparison = ['Baseline NN', 'Final NN']
rmse_values = [test_rmse_baseline, test_rmse_final]
r2_values = [test_r2_baseline, test_r2_final]

x = np.arange(len(models_comparison))
width = 0.35

bars1 = ax9.bar(x - width/2, rmse_values, width, label='RMSE (Wh)', alpha=0.8)
bars2 = ax9.bar(x + width/2, r2_values, width, label='R²', alpha=0.8)

ax9.set_xlabel('Model')
ax9.set_ylabel('Score')
ax9.set_title('Model Performance Comparison')
ax9.set_xticks(x)
ax9.set_xticklabels(models_comparison)
ax9.legend()
ax9.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax9.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Neural Network Analysis Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('nn_analysis_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("Visualization dashboard saved as 'nn_analysis_dashboard.png'")

In [ ]:
# --- 9. TIME SERIES ERROR ANALYSIS ---
print("\nPerforming time series error analysis...")

# Create time series analysis DataFrame
test_dates = df.loc[y_test.index, 'date']
ts_analysis = pd.DataFrame({
    'date': test_dates,
    'actual': y_test.values,
    'predicted': y_test_pred_final,
    'hour': test_dates.dt.hour,
    'day_of_week': test_dates.dt.dayofweek,
    'month': test_dates.dt.month,
    'error': y_test.values - y_test_pred_final,
    'abs_error': np.abs(y_test.values - y_test_pred_final),
    'percentage_error': np.abs((y_test.values - y_test_pred_final) / y_test.values) * 100
})

# Remove infinite values from percentage error
ts_analysis = ts_analysis.replace([np.inf, -np.inf], np.nan).dropna()

# Analysis by time period
print("\n--- Error Analysis by Time Period ---")

# Hourly analysis
hourly_stats = ts_analysis.groupby('hour').agg({
    'abs_error': ['mean', 'std', 'count'],
    'percentage_error': 'mean',
    'actual': 'mean'
}).round(2)

print("\nWorst prediction hours (highest MAE):")
worst_hours = hourly_stats['abs_error']['mean'].sort_values(ascending=False).head()
for hour, error in worst_hours.items():
    print(f"  Hour {hour:02d}: {error:.1f} Wh MAE")

# Daily analysis
daily_stats = ts_analysis.groupby('day_of_week').agg({
    'abs_error': ['mean', 'std'],
    'actual': 'mean'
}).round(2)

print("\nWorst prediction days (highest MAE):")
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
worst_days = daily_stats['abs_error']['mean'].sort_values(ascending=False).head()
for day, error in worst_days.items():
    print(f"  {day_names[day]}: {error:.1f} Wh MAE")

# Monthly analysis (if data spans multiple months)
if ts_analysis['month'].nunique() > 1:
    monthly_stats = ts_analysis.groupby('month').agg({
        'abs_error': ['mean', 'std'],
        'actual': 'mean'
    }).round(2)
    
    print("\nError by month:")
    for month in sorted(monthly_stats.index):
        error = monthly_stats.loc[month, ('abs_error', 'mean')]
        actual = monthly_stats.loc[month, ('actual', 'mean')]
        print(f"  Month {month}: {error:.1f} Wh MAE, {actual:.1f} Wh actual mean")

# Create time series error plot
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Error by hour
axes[0, 0].plot(hourly_stats.index, hourly_stats['abs_error']['mean'], 
               marker='o', linewidth=2)
axes[0, 0].fill_between(hourly_stats.index,
                       hourly_stats['abs_error']['mean'] - hourly_stats['abs_error']['std'],
                       hourly_stats['abs_error']['mean'] + hourly_stats['abs_error']['std'],
                       alpha=0.2)
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Mean Absolute Error (Wh)')
axes[0, 0].set_title('Prediction Error by Hour')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticks(range(0, 24, 3))

# 2. Error by day
axes[0, 1].bar(range(len(daily_stats)), daily_stats['abs_error']['mean'],
               yerr=daily_stats['abs_error']['std'], capsize=5, alpha=0.7)
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Mean Absolute Error (Wh)')
axes[0, 1].set_title('Prediction Error by Day')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels([name[:3] for name in day_names])
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Actual vs Predicted over time (sample of 200 points for clarity)
sample_size = min(200, len(ts_analysis))
sample_idx = np.random.choice(len(ts_analysis), sample_size, replace=False)
sample = ts_analysis.iloc[sample_idx].sort_values('date')

axes[1, 0].plot(sample['date'], sample['actual'], 'b-', alpha=0.7, label='Actual', linewidth=1)
axes[1, 0].plot(sample['date'], sample['predicted'], 'r-', alpha=0.7, label='Predicted', linewidth=1)
axes[1, 0].fill_between(sample['date'], sample['actual'], sample['predicted'],
                       alpha=0.2, color='gray', label='Error')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Energy (Wh)')
axes[1, 0].set_title('Actual vs Predicted (Sample)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Error distribution by actual consumption
axes[1, 1].scatter(ts_analysis['actual'], ts_analysis['abs_error'], alpha=0.5, s=10)
axes[1, 1].set_xlabel('Actual Consumption (Wh)')
axes[1, 1].set_ylabel('Absolute Error (Wh)')
axes[1, 1].set_title('Error vs Actual Consumption')
axes[1, 1].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(ts_analysis['actual'], ts_analysis['abs_error'], 1)
p = np.poly1d(z)
x_range = np.linspace(ts_analysis['actual'].min(), ts_analysis['actual'].max(), 100)
axes[1, 1].plot(x_range, p(x_range), "r--", alpha=0.8, 
               label=f'Trend: y={z[0]:.3f}x+{z[1]:.1f}')
axes[1, 1].legend()

plt.suptitle('Time Series Error Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('time_series_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Time series analysis saved as 'time_series_error_analysis.png'")

In [ ]:
# --- 10. PERFORMANCE METRICS EXPORT ---
print("\nExporting performance metrics...")

# Create comprehensive metrics DataFrame
metrics_data = {
    'metric': ['RMSE (Wh)', 'MAE (Wh)', 'MAPE (%)', 'R² Score', 
               'Training Time (epochs)', 'Model Parameters', 'MAE/Mean Ratio'],
    'baseline_nn': [
        test_rmse_baseline,
        test_mae_baseline,
        test_mape_baseline,
        test_r2_baseline,
        len(history_baseline.history['loss']),
        baseline_model.count_params(),
        test_mae_baseline / y_test.mean()
    ],
    'final_nn': [
        test_rmse_final,
        test_mae_final,
        test_mape_final,
        test_r2_final,
        len(history_final.history['loss']),
        final_model.count_params(),
        test_mae_final / y_test.mean()
    ],
    'improvement_percent': [
        (test_rmse_baseline - test_rmse_final) / test_rmse_baseline * 100,
        (test_mae_baseline - test_mae_final) / test_mae_baseline * 100,
        (test_mape_baseline - test_mape_final) / test_mape_baseline * 100,
        (test_r2_final - test_r2_baseline) / abs(test_r2_baseline) * 100,
        (len(history_baseline.history['loss']) - len(history_final.history['loss'])) / len(history_baseline.history['loss']) * 100,
        (final_model.count_params() - baseline_model.count_params()) / baseline_model.count_params() * 100,
        ((test_mae_baseline - test_mae_final) / y_test.mean()) * 100
    ]
}

metrics_df = pd.DataFrame(metrics_data)
metrics_df['baseline_nn'] = metrics_df['baseline_nn'].round(3)
metrics_df['final_nn'] = metrics_df['final_nn'].round(3)
metrics_df['improvement_percent'] = metrics_df['improvement_percent'].round(1)

print("\n=== PERFORMANCE METRICS SUMMARY ===")
print(metrics_df.to_string(index=False))

# Export to CSV
metrics_df.to_csv('neural_network_metrics.csv', index=False)

# Export detailed results
detailed_results = {
    'model_name': ['Baseline Neural Network', 'Final Neural Network'],
    'test_rmse': [test_rmse_baseline, test_rmse_final],
    'test_mae': [test_mae_baseline, test_mae_final],
    'test_mape': [test_mape_baseline, test_mape_final],
    'test_r2': [test_r2_baseline, test_r2_final],
    'training_epochs': [len(history_baseline.history['loss']), len(history_final.history['loss'])],
    'model_params': [baseline_model.count_params(), final_model.count_params()],
    'final_val_loss': [history_baseline.history['val_loss'][-1], history_final.history['val_loss'][-1]],
    'final_val_mae': [history_baseline.history['val_mae'][-1], history_final.history['val_mae'][-1]]
}

detailed_df = pd.DataFrame(detailed_results)
detailed_df.to_csv('detailed_nn_results.csv', index=False)

# Export feature importance
importance_df.to_csv('feature_importance_scores.csv', index=False)

# Export cross-validation results
cv_results_df.to_csv('cross_validation_results.csv', index=False)

print("\nFiles exported:")
print("  - neural_network_metrics.csv")
print("  - detailed_nn_results.csv")
print("  - feature_importance_scores.csv")
print("  - cross_validation_results.csv")

# Print final summary
print("\n" + "="*60)
print("NEURAL NETWORK ANALYSIS COMPLETE")
print("="*60)
print(f"Final Model Test RMSE: {test_rmse_final:.2f} Wh")
print(f"Final Model Test R²: {test_r2_final:.3f}")
print(f"Improvement over Baseline: {(test_rmse_baseline - test_rmse_final)/test_rmse_baseline:.2%}")
print(f"Cross-Validation RMSE: {mean_rmse:.2f} ± {std_rmse:.2f} Wh")
print(f"Most Important Feature: {importance_df.iloc[0]['feature']}")
print("="*60)